# TPC-H Q2 1GPU vs 4GPU Analysis

This notebook visualizes the Q2 multi-GPU experiment output generated under `mgpu_speedup_runs/q2_20260623_054837`.

Graphs included:
- Runtime per iteration: 1GPU vs 4GPU
- Per-iteration speedup
- GPU dispatch distribution in the 4GPU run
- Local / remote GPU / host input bytes before task preparation


In [ ]:
from pathlib import Path
import re
from collections import Counter, defaultdict

import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

RUN_DIR = Path('/home/dy1013/sirius_multi-gpu/mgpu_speedup_runs/q2_20260623_054837')
ONE_GPU_CSV = RUN_DIR / 'q2_1gpu/csv/runtimes.csv'
FOUR_GPU_CSV = RUN_DIR / 'q2_4gpu/csv/runtimes.csv'
FOUR_GPU_LOG = RUN_DIR / 'q2_4gpu/log_dir/sirius_2026-06-23.log'
ONE_GPU_LOG = RUN_DIR / 'q2_1gpu/log_dir/sirius_2026-06-23.log'
OUT_DIR = RUN_DIR / 'graphs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_DIR


In [ ]:
one = pd.read_csv(ONE_GPU_CSV).rename(columns={'runtime_s': 'runtime_1gpu_s'})
four = pd.read_csv(FOUR_GPU_CSV).rename(columns={'runtime_s': 'runtime_4gpu_s'})
df = one[['iteration', 'runtime_1gpu_s']].merge(four[['iteration', 'runtime_4gpu_s']], on='iteration')
df['speedup_1gpu_over_4gpu'] = df['runtime_1gpu_s'] / df['runtime_4gpu_s']
warm = df[df['iteration'] >= 2]
summary = pd.DataFrame([{
    'metric': 'warm_median_excluding_iter0_iter1',
    'runtime_1gpu_s': warm['runtime_1gpu_s'].median(),
    'runtime_4gpu_s': warm['runtime_4gpu_s'].median(),
    'speedup_1gpu_over_4gpu': warm['runtime_1gpu_s'].median() / warm['runtime_4gpu_s'].median(),
}])
display(df)
display(summary)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df['iteration'], df['runtime_1gpu_s'], marker='o', label='1 GPU')
ax.plot(df['iteration'], df['runtime_4gpu_s'], marker='o', label='4 GPUs')
ax.set_title('TPC-H Q2 Runtime: 1GPU vs 4GPU')
ax.set_xlabel('Iteration')
ax.set_ylabel('Runtime (s)')
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / 'q2_runtime_1gpu_vs_4gpu.png', dpi=160)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#2ca02c' if v >= 1.0 else '#d62728' for v in df['speedup_1gpu_over_4gpu']]
ax.bar(df['iteration'], df['speedup_1gpu_over_4gpu'], color=colors)
ax.axhline(1.0, color='black', linewidth=1)
ax.set_title('TPC-H Q2 Per-Iteration Speedup: 1GPU runtime / 4GPU runtime')
ax.set_xlabel('Iteration')
ax.set_ylabel('Speedup')
ax.text(0.01, 0.95, f"Warm median speedup: {summary.loc[0, 'speedup_1gpu_over_4gpu']:.3f}x",
        transform=ax.transAxes, va='top')
fig.tight_layout()
fig.savefig(OUT_DIR / 'q2_speedup_bars.png', dpi=160)
plt.show()


In [ ]:
def parse_locality_log(path: Path):
    dispatch_counts = Counter()
    prepare = defaultdict(lambda: {'tasks': 0, 'input_bytes': 0, 'local_bytes': 0, 'remote_gpu_bytes': 0, 'host_bytes': 0, 'disk_bytes': 0})
    dispatch_re = re.compile(r'\[locality-audit\] dispatch .*actual_gpu=(\d+)')
    prepare_re = re.compile(
        r'\[locality-audit\] prepare_before .*actual_gpu=(\d+).*'
        r'input_bytes=(\d+) local_bytes=(\d+) remote_gpu_bytes=(\d+) host_bytes=(\d+) disk_bytes=(\d+)'
    )
    for line in path.read_text(errors='replace').splitlines():
        m = dispatch_re.search(line)
        if m:
            dispatch_counts[int(m.group(1))] += 1
        m = prepare_re.search(line)
        if m:
            gpu = int(m.group(1))
            input_b, local_b, remote_b, host_b, disk_b = map(int, m.groups()[1:])
            prepare[gpu]['tasks'] += 1
            prepare[gpu]['input_bytes'] += input_b
            prepare[gpu]['local_bytes'] += local_b
            prepare[gpu]['remote_gpu_bytes'] += remote_b
            prepare[gpu]['host_bytes'] += host_b
            prepare[gpu]['disk_bytes'] += disk_b
    dispatch_df = pd.DataFrame(sorted(dispatch_counts.items()), columns=['gpu', 'dispatches'])
    prepare_df = pd.DataFrame([{'gpu': gpu, **vals} for gpu, vals in sorted(prepare.items())])
    for col in ['input_bytes', 'local_bytes', 'remote_gpu_bytes', 'host_bytes', 'disk_bytes']:
        prepare_df[col.replace('_bytes', '_gb')] = prepare_df[col] / 1e9
    return dispatch_df, prepare_df

dispatch4, prepare4 = parse_locality_log(FOUR_GPU_LOG)
dispatch1, prepare1 = parse_locality_log(ONE_GPU_LOG)
display(dispatch4)
display(prepare4)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(dispatch4['gpu'].astype(str), dispatch4['dispatches'], color='#4c78a8')
ax.set_title('4GPU Q2 Dispatch Count by Actual GPU')
ax.set_xlabel('GPU')
ax.set_ylabel('Dispatched tasks')
for i, v in enumerate(dispatch4['dispatches']):
    ax.text(i, v, str(v), ha='center', va='bottom')
fig.tight_layout()
fig.savefig(OUT_DIR / 'q2_4gpu_dispatch_counts.png', dpi=160)
plt.show()


In [ ]:
plot_df = prepare4.set_index('gpu')[['local_gb', 'remote_gpu_gb', 'host_gb', 'disk_gb']]
fig, ax = plt.subplots(figsize=(10, 5))
plot_df.plot(kind='bar', stacked=True, ax=ax, color=['#59a14f', '#e15759', '#f28e2b', '#9c755f'])
ax.set_title('4GPU Q2 Input Locality Before Prepare')
ax.set_xlabel('Actual GPU')
ax.set_ylabel('Input bytes by source (GB)')
ax.legend(['Local GPU', 'Remote GPU', 'Host', 'Disk'], loc='upper right')
fig.tight_layout()
fig.savefig(OUT_DIR / 'q2_4gpu_locality_bytes.png', dpi=160)
plt.show()


In [ ]:
total4 = prepare4[['input_bytes', 'local_bytes', 'remote_gpu_bytes', 'host_bytes', 'disk_bytes']].sum()
total1 = prepare1[['input_bytes', 'local_bytes', 'remote_gpu_bytes', 'host_bytes', 'disk_bytes']].sum()
compare = pd.DataFrame({'1GPU_GB': total1 / 1e9, '4GPU_GB': total4 / 1e9})
compare.loc['local_ratio'] = [total1['local_bytes'] / total1['input_bytes'], total4['local_bytes'] / total4['input_bytes']]
compare.loc['remote_ratio'] = [0.0, total4['remote_gpu_bytes'] / total4['input_bytes']]
compare.loc['host_ratio'] = [total1['host_bytes'] / total1['input_bytes'], total4['host_bytes'] / total4['input_bytes']]
display(compare)


## Interpretation

For this run, Q2 uses all four GPUs, but warm median speedup is close to 1.0x. The locality graph shows a large amount of remote GPU and host-sourced input before task preparation in the 4GPU case. This supports the motivation that naive multi-GPU task distribution can introduce cross-GPU/host movement that cancels out expected parallel speedup.
